<a href="https://colab.research.google.com/github/kristinaswan/pet_projects/blob/main/st_collision_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Часть II – Анализ качества данных

Осуществим проверку качества данных:

• найдем дубликаты source_object_id внутри одного набора,\
• выявим некорректные даты,\
• выявим пропуски,\
• проверим валидность геометрии,\
• найдем долю проблемных записей.

Результат:\
• таблица метрик,\
• перечень выявленных проблем,\
• краткие выводы.

In [ ]:
# Импортируем необходимые библиотеки
import pandas as pd
import geopandas as gpd
from shapely.wkt import loads
from shapely.geometry import Polygon, MultiPolygon

In [ ]:
pd.reset_option('display.max_colwidth')

In [ ]:
# Прочитаем файл с датасетом
df = pd.read_csv('/content/street_cases.csv', sep='$')
df.head(20)

,dataset_code,source_object_id,work_type,status,start_dt,end_dt,contractor,priority,geom_a_wkt,geom_b_wkt,geom_c_wkt,address_raw,comment
0,A,A-001,Благоустройство: МАФ,done,2026-02-27,2026-03-09,ГУП Доринвест,2.0,MULTIPOLYGON (((37.5422269787993 55.7084730940...,NaN,NaN,Улица Косыгина,NaN
1,A,A-002,Благоустройство: МАФ,canceled,2026-03-10,2026-03-20,ООО Рога и Копыта,1.0,MULTIPOLYGON (((37.5353354427581 55.7161775055...,NaN,NaN,ул. Косыгина,срочно
2,A,A-003,Благоустройство: МАФ,in_progress,2026-03-14,2026-03-24,NaN,2.0,MULTIPOLYGON (((37.5350214688055 55.7161804240...,NaN,NaN,"ул Косыгина, Москва",уточнить границы
3,A,A-003,Благоустройство: МАФ,canceled,2026-03-20,2026-04-03,АО ИнжСтрой,3.0,MULTIPOLYGON (((37.5352782511097 55.7257250138...,NaN,NaN,Улица Косыгина,срочно
4,A,A-005,Благоустройство: покрытие,canceled,2026-03-29,2026-04-08,ООО Рога и Копыта,3.0,MULTIPOLYGON (((37.5356828095321 55.7190568533...,NaN,NaN,Косыгина ул,уточнить границы
5,A,A-006,Благоустройство: МАФ,planned,2026-04-03,2026-05-03,NaN,1.0,NaN,NaN,NaN,УЛ.КОСЫГИНА,уточнить границы
6,B,B-001,Инжсети: связь,in_progress,2026-03-03,2026-03-24,ООО Рога и Копыта,1.0,NaN,"MULTIPOLYGON (((37.54273761 55.70971716, 37.54...",NaN,Kosygina st.,срочно
7,B,B-002,Инжсети: теплосеть,planned,2026-03-08,2026-03-05,NaN,3.0,NaN,MULTIPOLYGON (((37.5353308755148 55.7161606398...,NaN,"ул Косыгина, Москва",уточнить границы
8,B,B-003,Инжсети: водопровод,in_progress,2026-03-14,2026-04-04,АО ИнжСтрой,NaN,NaN,MULTIPOLYGON (((37.5353308755148 55.7161606398...,NaN,Kosygina st.,NaN
9,B,B-004,Инжсети: связь,in_progress,2026-03-24,2026-04-07,ГУП Доринвест,4.0,NaN,"MULTIPOLYGON (((37.54273761 55.70971716, 37.54...",NaN,"ул Косыгина, Москва",уточнить границы


In [ ]:
# Получим основную информацию о таблице с помощью метода info()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   dataset_code      15 non-null     object 
 1   source_object_id  15 non-null     object 
 2   work_type         15 non-null     object 
 3   status            15 non-null     object 
 4   start_dt          15 non-null     object 
 5   end_dt            14 non-null     object 
 6   contractor        12 non-null     object 
 7   priority          13 non-null     float64
 8   geom_a_wkt        5 non-null      object 
 9   geom_b_wkt        6 non-null      object 
 10  geom_c_wkt        3 non-null      object 
 11  address_raw       15 non-null     object 
 12  comment           12 non-null     object 
dtypes: float64(1), object(12)
memory usage: 1.7+ KB


### Дубликаты source_object_id внутри одного набора

In [ ]:
# Найдем дубликаты source_object_id внутри одного набора (dataset_code)
duplicates = df.duplicated(subset=['dataset_code', 'source_object_id'], keep=False)

In [ ]:
# Найдем количество дубликатов
num_duplicates = duplicates.sum()
print(f"Количество дубликатов: {num_duplicates}")

Количество дубликатов: 2


In [ ]:
# Выведем найденные дубликаты
if num_duplicates > 0:
  print("Найдены следующие дубликаты:")
  display(df[duplicates])
else:
  print("Дубликаты не найдены.")

Найдены следующие дубликаты:


,dataset_code,source_object_id,work_type,status,start_dt,end_dt,contractor,priority,geom_a_wkt,geom_b_wkt,geom_c_wkt,address_raw,comment
2,A,A-003,Благоустройство: МАФ,in_progress,2026-03-14,2026-03-24,NaN,2.0,MULTIPOLYGON (((37.5350214688055 55.7161804240...,NaN,NaN,"ул Косыгина, Москва",уточнить границы
3,A,A-003,Благоустройство: МАФ,canceled,2026-03-20,2026-04-03,АО ИнжСтрой,3.0,MULTIPOLYGON (((37.5352782511097 55.7257250138...,NaN,NaN,Улица Косыгина,срочно


### Некорректные даты

In [ ]:
# Найдем некорректные даты (где начало работ позже их окончания)
invalid_dates  = (df['start_dt'] > df['end_dt'])

In [ ]:
# Найдем количество некорректных дат (где начало работ позже их окончания)
num_invalid_dates = invalid_dates.sum()
print(f"Количество некорректных дат (где начало работ позже их окончания): {num_invalid_dates}")

Количество некорректных дат (где начало работ позже их окончания): 1


In [ ]:
# Выведем найденные некорректные даты (где начало работ позже их окончания)
if num_invalid_dates > 0:
  print("Найдены следующие некорректные даты (где начало работ позже их окончания):")
  display(df[invalid_dates])
else:
  print("Некорректные даты не найдены.")

Найдены следующие некорректные даты (где начало работ позже их окончания):


,dataset_code,source_object_id,work_type,status,start_dt,end_dt,contractor,priority,geom_a_wkt,geom_b_wkt,geom_c_wkt,address_raw,comment
7,B,B-002,Инжсети: теплосеть,planned,2026-03-08,2026-03-05,NaN,3.0,NaN,MULTIPOLYGON (((37.5353308755148 55.7161606398...,NaN,"ул Косыгина, Москва",уточнить границы


### Пропуски в значениях

В исходном датасете представлена информация о трех разных наборах данных (три разных источника/типа работ), где для каждого представлена своя геометрия (geom_a_wkt, geom_b_wkt, geom_c_wkt).\
Чтобы корректно проверить пропуски в значениях и случайно не учесть такие, где значений по логике быть не должно, для каждого набора данных (A, B, C) выбирем соответствующие строки и проверим геометрию в нужной колонке.

Рассмторим пропуски в ключевых полях:\
• dataset_code\
• source_object_id\
• work_type\
• status\
• start_dt\
• end_dt\
• priority\
• address_raw\
• геометрия в WKT.

Поля contractor и comment рассматривать не будем.

In [ ]:
# Найдем пропуски в значениях
cols_to_exclude_a = ['geom_b_wkt', 'geom_c_wkt', 'contractor', 'comment']
cols_to_exclude_b = ['geom_a_wkt', 'geom_c_wkt', 'contractor', 'comment']
cols_to_exclude_c = ['geom_a_wkt', 'geom_b_wkt', 'contractor', 'comment']

missing_in_a = df.query("dataset_code == 'A'").drop(cols_to_exclude_a, axis=1).isnull().sum()
missing_in_b = df.query("dataset_code == 'B'").drop(cols_to_exclude_b, axis=1).isnull().sum()
missing_in_c = df.query("dataset_code == 'C'").drop(cols_to_exclude_c, axis=1).isnull().sum()

In [ ]:
# Выведем результат проверки
print(f'Пропуски в значениях в датасете A: \n{missing_in_a}')
print(f'\nПропуски в значениях в датасете B: \n{missing_in_b}')
print(f'\nПропуски в значениях в датасете C: \n{missing_in_c}')

Пропуски в значениях в датасете A: 
dataset_code        0
source_object_id    0
work_type           0
status              0
start_dt            0
end_dt              0
priority            0
geom_a_wkt          1
address_raw         0
dtype: int64

Пропуски в значениях в датасете B: 
dataset_code        0
source_object_id    0
work_type           0
status              0
start_dt            0
end_dt              0
priority            1
geom_b_wkt          0
address_raw         0
dtype: int64

Пропуски в значениях в датасете C: 
dataset_code        0
source_object_id    0
work_type           0
status              0
start_dt            0
end_dt              1
priority            1
geom_c_wkt          0
address_raw         0
dtype: int64


### Валидность геометрии на целостность

In [ ]:
# Создадим функцию
def check_geometry_validity(wkt):
  """
  Функция проверки валидности геометрии на целостность.
  """
  try:
    geom = loads(wkt)
    return geom.is_valid
  except:
    return False

In [ ]:
# Проверим геометрии для каждого набора данных
valid_geom_a = df.query("dataset_code == 'A'")['geom_a_wkt'].apply(check_geometry_validity)
valid_geom_b = df.query("dataset_code == 'B'")['geom_b_wkt'].apply(check_geometry_validity)
valid_geom_c = df.query("dataset_code == 'C'")['geom_c_wkt'].apply(check_geometry_validity)

In [ ]:
# Выведем результат проверки
print(f'Валидность геометрии на целостность в датасете А: \n{valid_geom_a}')
print(f'\nВалидность геометрии на целостность в датасете B: \n{valid_geom_b}')
print(f'\nВалидность геометрии на целостность в датасете C: \n{valid_geom_c}')

Валидность геометрии на целостность в датасете А: 
0     True
1     True
2     True
3     True
4     True
5    False
Name: geom_a_wkt, dtype: bool

Валидность геометрии на целостность в датасете B: 
6      True
7      True
8      True
9      True
10    False
11     True
Name: geom_b_wkt, dtype: bool

Валидность геометрии на целостность в датасете C: 
12    True
13    True
14    True
Name: geom_c_wkt, dtype: bool


In [ ]:
# Выведем результат проверки
print(f'Количество недействительных геометрий в датасете А (в поле geom_a_wkt): {sum(~valid_geom_a)}')
print(f'Количество недействительных геометрий в датасете B (в поле geom_b_wkt): {sum(~valid_geom_b)}')
print(f'Количество недействительных геометрий в датасете C (в поле geom_c_wkt): {sum(~valid_geom_c)}')


Количество недействительных геометрий в датасете А (в поле geom_a_wkt): 1
Количество недействительных геометрий в датасете B (в поле geom_b_wkt): 1
Количество недействительных геометрий в датасете C (в поле geom_c_wkt): 0


### Доля проблемных записей

В исходном датасете представлена информация о трех разных наборах данных (три разных источника/типа работ), где для каждого представлена своя геометрия (geom_a_wkt, geom_b_wkt, geom_c_wkt).\
Чтобы корректно проверить пропуски в значениях и случайно не учесть такие, где значений по логике быть не должно, для каждого набора данных (A, B, C) выбирем соответствующие строки и проверим геометрию в нужной колонке.

Рассмторим пропуски в ключевых полях:\
• dataset_code\
• source_object_id\
• work_type\
• status\
• start_dt\
• end_dt\
• priority\
• address_raw\
• геометрия в WKT.

Поля contractor и comment рассматривать не будем.

In [ ]:
# 1. Найдем дубликаты source_object_id внутри одного набора (dataset_code)
# duplicates = df.duplicated(subset=['dataset_code', 'source_object_id'], keep=False)

# 2. Найдем некорректные даты (где начало работ позже их окончания)
# invalid_dates  = (df['start_dt'] > df['end_dt'])

# 3. Найдем пропуски в значениях
# cols_to_exclude_a = ['geom_b_wkt', 'geom_c_wkt', 'contractor', 'comment']
# cols_to_exclude_b = ['geom_a_wkt', 'geom_c_wkt', 'contractor', 'comment']
# cols_to_exclude_c = ['geom_a_wkt', 'geom_b_wkt', 'contractor', 'comment']

# Подсчитаем пропуски в колонках, соответствующие каждому набору данных
missing_in_a = df.query("dataset_code == 'A'").drop(cols_to_exclude_a, axis=1).isnull().any(axis=1)
missing_in_b = df.query("dataset_code == 'B'").drop(cols_to_exclude_b, axis=1).isnull().any(axis=1)
missing_in_c = df.query("dataset_code == 'C'").drop(cols_to_exclude_c, axis=1).isnull().any(axis=1)

# 4. Валидность геометрии на целостность
# # Создадим функцию
# def check_geometry_validity(wkt):
#   """
#   Функция проверки валидности геометрии на целостность.
#   """
#   try:
#     geom = loads(wkt)
#     return geom.is_valid
#   except:
#     return False

# Проверим геометрии для каждого набора данных
# valid_geom_a = df.query("dataset_code == 'A'")['geom_a_wkt'].apply(check_geometry_validity)
# valid_geom_b = df.query("dataset_code == 'B'")['geom_b_wkt'].apply(check_geometry_validity)
# valid_geom_c = df.query("dataset_code == 'C'")['geom_c_wkt'].apply(check_geometry_validity)

# Найдем неверные геометрии
invalid_geom_a = ~valid_geom_a
invalid_geom_b = ~valid_geom_b
invalid_geom_c = ~valid_geom_c

# 5. Объединим все условия в единую маску
problem_mask = (
      duplicates |                                                                # Дубликаты
      invalid_dates |                                                             # Некорректные даты
      missing_in_a |                                                              # Пропуски в датасете A
      missing_in_b |                                                              # Пропуски в датасете B
      missing_in_c |                                                              # Пропуски в датасете C
      invalid_geom_a |                                                            # Неверная геометрия в датасете A
      invalid_geom_b |                                                            # Неверная геометрия в датасете B
      invalid_geom_c                                                              # Неверная геометрия в датасете C
)

In [ ]:
# 6. Подсчитаем все проблемные записи
problematic_rows_unique = problem_mask.sum()

In [ ]:
# 7. Рассчитаем долю проблемных записей
total_records = len(df)
problem_ratio = problematic_rows_unique / total_records

In [ ]:
# 8. Выведем полученный результат
print(f"Доля проблемных записей: {problem_ratio}")

Доля проблемных записей: 0.5333333333333333


### Результат

In [ ]:
# Соберем данные для таблицы метрик
metrics_data = {
      'Название показателя': [
          'Общее количество записей',
          'Количество дубликатов',
          'Количество некорректных дат',
          'Количество пропусков',
          'Количество записей с неверной геометрией',
          'Доля проблемных записей'
      ],
      'Значение': [
          total_records,                                                          # Общее количество записей
          duplicates.sum(),                                                       # Количество дубликатов
          invalid_dates.sum(),                                                    # Количество некорректных дат
          missing_in_a.sum() + missing_in_b.sum() + missing_in_c.sum(),           # Количество пропусков
          invalid_geom_a.sum() + invalid_geom_b.sum() + invalid_geom_c.sum(),     # Количество записей с неверной геометрией
          f'{problem_ratio}'                                                      # Доля проблемных записей
      ]
}

In [ ]:
# Создадим DataFrame
metrics_table = pd.DataFrame(metrics_data)

In [ ]:
# Выведем таблицу метрик
print(metrics_table)

                        Название показателя            Значение
0                  Общее количество записей                  15
1                     Количество дубликатов                   2
2               Количество некорректных дат                   1
3                      Количество пропусков                   4
4  Количество записей с неверной геометрией                   2
5                   Доля проблемных записей  0.5333333333333333


Перечень выявленных проблем:

*   Дубликаты в поле source_object_id в датасете A;
*   Некорректные даты (где начало работ позже их окончания);
*   Пропуски в ключевых полях;
*   Ошибки геометрии - отсутствие данных и неправильные координаты.

Краткие выводы:

В результате анализа был выявлен ряд проблем, которые снижают качество анализа данных. Так,:

*   Найденные дубликаты говорят о необходимости создания дополнительной проверки на уникальность данных;
*   Некорректные даты указывают на слабые процедуры проверки данных на входе;
*   Наличие пропусков в ключевых полях создают сложности для обработки и анализа данных;
*   Записи с неверной и пропущенной геометирей ограничивают возможности пространственного анализа.

Рекомендации:

1.   Устранить дубликаты;
2.   Проверить и устранить некорректные даты;
3.   Создать механизмы/алгоритмы проверки полноты данных;
4.   Определить стандарты загрузки геометрий и ввести проверку на их корректность.

## Часть III – Пространственный анализ

1. Сформируем три пространственных слоя из CSV.
2. Проверим корректность геометрии и CRS.
3. Найдем коллизии между:\
o A × B\
o A × C\
o B × C

Для каждой пары объектов определим:\
• факт пространственного пересечения,\
• факт пересечения по времени,\
• количество дней перекрытия,\
• метрику «силы конфликта» (с обоснованием).

Результат:\
• таблица всех выявленных коллизий,\
• описание методики расчета.

### Предобработка данных

In [ ]:
# Преобразуем строки в объекты datetime
df['start_dt'] = pd.to_datetime(df['start_dt'])
df['end_dt'] = pd.to_datetime(df['end_dt'])

In [ ]:
# Приравняем значения Nan в поле priorities к значению "1"
df['priority'] = df['priority'].fillna(1)

In [ ]:
# Получим основную информацию о таблице с помощью метода info()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   dataset_code      15 non-null     object        
 1   source_object_id  15 non-null     object        
 2   work_type         15 non-null     object        
 3   status            15 non-null     object        
 4   start_dt          15 non-null     datetime64[ns]
 5   end_dt            14 non-null     datetime64[ns]
 6   contractor        12 non-null     object        
 7   priority          15 non-null     float64       
 8   geom_a_wkt        5 non-null      object        
 9   geom_b_wkt        6 non-null      object        
 10  geom_c_wkt        3 non-null      object        
 11  address_raw       15 non-null     object        
 12  comment           12 non-null     object        
dtypes: datetime64[ns](2), float64(1), object(10)
memory usage: 1.7+ KB


In [ ]:
# Оставим только те работы для анализа, которые имеют статус  “in_progress” и “planned”
df_actual = df.query("(status == 'in_progress') | (status == 'planned')")

### Формирование пространственных слоев

In [ ]:
# Импортируем необходимые библиотеки
from shapely.errors import WKTReadingError

# Создадим функцию
def safe_loads(x):
  """
  Функция предназначена для безопасного преобразования геометрии.

  :param x: строка геометрия в формате WKT (полигоны и мультиполигоны).
  :return: объект геометрии, поддерживаемый библиотекой shapely. В случае неправильной геометрии вернётся None.
  """
  try:
      return loads(x)
  except WKTReadingError:
      return None

/tmp/ipython-input-258/2770227694.py:2: FutureWarning: WKTReadingError is deprecated and will be removed in a future version. Use ShapelyError instead (functions previously raising {name} will now raise a ShapelyError instead).
  from shapely.errors import WKTReadingError


In [ ]:
# Во избежании ошибок, перед преобразованием проверим, является ли значение строкой
geom_a = df_actual['geom_a_wkt'].apply(lambda x: safe_loads(x) if isinstance(x, str) else None)
geom_b = df_actual['geom_b_wkt'].apply(lambda x: safe_loads(x) if isinstance(x, str) else None)
geom_c = df_actual['geom_c_wkt'].apply(lambda x: safe_loads(x) if isinstance(x, str) else None)

In [ ]:
# Исключим записи с None
valid_geom_a = geom_a.notnull()
valid_geom_b = geom_b.notnull()
valid_geom_c = geom_c.notnull()

In [ ]:
# Создадим отдельный GeoDataFrame для каждого набора работ (A, B и C)
gdf_a = gpd.GeoDataFrame(df_actual[valid_geom_a], geometry=geom_a[valid_geom_a])
gdf_b = gpd.GeoDataFrame(df_actual[valid_geom_b], geometry=geom_b[valid_geom_b])
gdf_c = gpd.GeoDataFrame(df_actual[valid_geom_c], geometry=geom_c[valid_geom_c])

In [ ]:
# На примере датасета A проверим правильность формирования пространственных слоев
# Получим основную информацию о таблице с помощью метода info()
gdf_a.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 1 entries, 2 to 2
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   dataset_code      1 non-null      object        
 1   source_object_id  1 non-null      object        
 2   work_type         1 non-null      object        
 3   status            1 non-null      object        
 4   start_dt          1 non-null      datetime64[ns]
 5   end_dt            1 non-null      datetime64[ns]
 6   contractor        0 non-null      object        
 7   priority          1 non-null      float64       
 8   geom_a_wkt        1 non-null      object        
 9   geom_b_wkt        0 non-null      object        
 10  geom_c_wkt        0 non-null      object        
 11  address_raw       1 non-null      object        
 12  comment           1 non-null      object        
 13  geometry          1 non-null      geometry      
dtypes: datetime64[ns](2), float

### Проверка на корректность геометрии и CRS

In [ ]:
# Проверим валидность геометрий
invalid_geom_a = ~gdf_a.geometry.is_valid
invalid_geom_b = ~gdf_b.geometry.is_valid
invalid_geom_c = ~gdf_c.geometry.is_valid

In [ ]:
# Выведем результаты проверки по каждому из датасетов
print(f"Количество некорректных геометрий в датасете A: {invalid_geom_a.sum()}")
print(f"Количество некорректных геометрий в датасете B: {invalid_geom_b.sum()}")
print(f"Количество некорректных геометрий в датасете C: {invalid_geom_c.sum()}")

Количество некорректных геометрий в датасете A: 0
Количество некорректных геометрий в датасете B: 0
Количество некорректных геометрий в датасете C: 0


In [ ]:
# Предварительно проверим наличие системы координат
print(gdf_a.crs)
print(gdf_b.crs)
print(gdf_c.crs)

None
None
None


In [ ]:
# Установим для каждого из набора данных систему координат WGS84
gdf_a.crs = "EPSG:4326"
gdf_b.crs = "EPSG:4326"
gdf_c.crs = "EPSG:4326"

In [ ]:
# Проверим правильность преобразования системы координат в наборах данных
print(gdf_a.crs)
print(gdf_b.crs)
print(gdf_c.crs)

EPSG:4326
EPSG:4326
EPSG:4326


### Нахождение коллизий между парами

3. Найдем коллизии между:\
o A × B\
o A × C\
o B × C

Для каждой пары объектов определим:\
• факт пространственного пересечения,\
• факт пересечения по времени,\
• количество дней перекрытия,\
• метрику «силы конфликта» (с обоснованием).

In [ ]:
# Импортируем необходимые библиотеки
# import pandas as pd
from datetime import timedelta
from shapely.ops import transform
from functools import partial
import pyproj

In [ ]:
# Создадим функцию
def to_meters(geom):
  """
  Функция для перевода геометрии в метрическую систему координат.
  """
  project = partial(pyproj.transform, pyproj.Proj(init='epsg:4326'),pyproj.Proj(init='epsg:3857'))
  return transform(project, geom)

def calculate_overlap_days(start1, end1, start2, end2):
  """
  Функция для расчета дней перекрытия.
  """
  latest_start = max(start1, start2)
  earliest_end = min(end1, end2)
  delta = (earliest_end - latest_start).days
  return max(delta, 0)

def conflict_strength(priorities, days_overlapping, ratio_area_overlap):
  """
  Функция для расчета силы конфликта.
  """
  max_priority = max(priorities)
  return max_priority * days_overlapping * ratio_area_overlap

def intersection_ratio(geom1, geom2):
  """
  Функция для расчета доли площади пересечения.
  """
  # Преобразуем геометрию в метрическую систему координат
  geom1_proj = to_meters(geom1)
  geom2_proj = to_meters(geom2)

  intersection = geom1_proj.intersection(geom2_proj)
  union_area = geom1_proj.union(geom2_proj).area  # Общая площадь
  percent_overlap = (intersection.area / union_area) if union_area > 0 else 0
  return round(percent_overlap, 2)

def determine_criticality(overlap_days, ratio_area_overlap, priorities):
  """
  Функция для определения уровня критичности коллизии.
  """
  max_priority = max(priorities)
  min_area_overlap_threshold = 0.5  # половина минимальной площади
  quarter_area_overlap_threshold = 0.25  # четверть минимальной площади

  if ratio_area_overlap > min_area_overlap_threshold and overlap_days > 5 and max_priority >= 3.0:
    return 'high'
  elif quarter_area_overlap_threshold <= ratio_area_overlap <= min_area_overlap_threshold and 1 <= overlap_days <= 5 and max_priority == 2.0:
    return 'medium'
  else:
    return 'low'

# Проверим пространственные и временные коллизии
pairs = [(gdf_a, gdf_b), (gdf_a, gdf_c), (gdf_b, gdf_c)]

conflicts = []

for i, (layer1, layer2) in enumerate(pairs):
  # Внешний цикл по первому слою
  for _, row1 in layer1.iterrows():
    found_match = False    # Флаг, который покажет, была ли найдена коллизия

    # Цикл по второму слою
    for _, row2 in layer2.iterrows():
      # Пространственные пересечения
      intersects_spatially = row1.geometry.intersects(row2.geometry)

      # Временные пересечения
      intersects_temporally = calculate_overlap_days(row1['start_dt'], row1['end_dt'], row2['start_dt'], row2['end_dt'])

      # Количество дней перекрытия
      overlap_days = intersects_temporally

      # Доля перекрытия
      ratio_area_overlap = intersection_ratio(row1.geometry, row2.geometry) if intersects_spatially else 0

      # Сила конфликта
      strength = conflict_strength([row1['priority'], row2['priority']], overlap_days, ratio_area_overlap)

      # Добавляем приоритеты обеих работ
      priorities = [row1['priority'], row2['priority']]

      # Добавляем статус обеих работ
      statuses = [row1['status'], row2['status']]

      # Определяем уровень критичности
      level = determine_criticality(overlap_days, ratio_area_overlap, priorities)

      # Если есть пересечение по пространству или времени
      # if intersects_spatially or intersects_temporally > 0:
      conflicts.append({
          'pair': f"{row1['dataset_code']}-{row2['dataset_code']}",
          'source_ids': f"{row1['source_object_id']}-{row2['source_object_id']}",
          'priorities': ', '.join(map(str, priorities)),  # Преобразуем список в строку
          'statuses': ', '.join(map(str, statuses)),  # Преобразуем список в строку
          'spatial_intersection': intersects_spatially,
          'temporal_intersection': intersects_temporally > 0,
          'ratio_area_overlap': ratio_area_overlap,  # Добавляем процент перекрытия
          'overlap_days': overlap_days,
          'strength': strength,
          'level': level
      })
      found_match = True    # Установим флаг, что коллизия найдена

    # Если коллизия не найдена ни с одним объектом второго слоя, добавим пустую запись
    if not found_match:
      # Возьмем первое значение из second layer, чтобы заполнить source_id
      conflicts.append({
          'pair': f"{row1['dataset_code']}-{layer2.iloc[0]['dataset_code']}",
          'source_ids': f"{row1['source_object_id']}-{layer2.iloc[0]['source_object_id']}",
          'priorities': '',  # Пустой приоритет
          'statuses': '',  # Пустой статус
          'spatial_intersection': False,
          'ratio_area_overlap': 0,  # Процент перекрытия равен нулю
          'temporal_intersection': False,
          'overlap_days': 0,
          'strength': 0,
          'level': 'low'
      })

/usr/local/lib/python3.12/dist-packages/pyproj/crs/crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
/usr/local/lib/python3.12/dist-packages/pyproj/crs/crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)
/usr/local/lib/python3.12/dist-packages/shapely/ops.py:259: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgrading-to-pyproj-2-from-pyproj-1
  shell = type(geom.exterior

### Результат

In [ ]:
# Создадим итоговый DataFrame с результатами
conflicts_df = pd.DataFrame(conflicts)

In [ ]:
# Отобразим итоговый DataFrame с результатами
conflicts_df

,pair,source_ids,priorities,statuses,spatial_intersection,temporal_intersection,ratio_area_overlap,overlap_days,strength,level
0,A-B,A-003-B-001,"2.0, 1.0","in_progress, in_progress",False,True,0.00,10,0.00,low
1,A-B,A-003-B-002,"2.0, 3.0","in_progress, planned",True,False,0.62,0,0.00,low
2,A-B,A-003-B-003,"2.0, 1.0","in_progress, in_progress",True,True,0.62,10,12.40,low
3,A-B,A-003-B-004,"2.0, 4.0","in_progress, in_progress",False,False,0.00,0,0.00,low
4,A-B,A-003-B-006,"2.0, 2.0","in_progress, in_progress",True,False,0.00,0,0.00,low
5,A-C,A-003-C-003,"2.0, 1.0","in_progress, in_progress",True,True,0.62,9,11.16,low
6,B-C,B-001-C-003,"1.0, 1.0","in_progress, in_progress",False,True,0.00,10,0.00,low
7,B-C,B-002-C-003,"3.0, 1.0","planned, in_progress",True,False,1.00,0,0.00,low
8,B-C,B-003-C-003,"1.0, 1.0","in_progress, in_progress",True,True,1.00,9,9.00,low
9,B-C,B-004-C-003,"4.0, 1.0","in_progress, in_progress",False,False,0.00,0,0.00,low


Методика расчета:


*   Пространственное пересечение (spatial_intersection) было выявлено с помощью функции intersects() библиотеки shapely;
*   Временное пересечение (temporal_intersection) было рассчитано как разница между минимальным концом периода работ и максимальным началом периода работ. Если период является положительным, то пересечение есть;
*   Доля площади перекрытия (ratio_area_overlap) – это доля физического пересечения площадей;
*   Количество дней перекрытия (overlap_days) – это количество дней, когда периоды пересекаются;
*   Сила конфликта – это произведение максимального приоритета работ, доли площади перекрытия и количества дней перекрытия;
*   Критичность коллизий (level) – это уровень конфликта, основанный на сочетании нескольких факторов: продолжительности временного пересечения, размера пространсвенного перекрытия и уровеня приоритетов задействованных работ.



## Часть IV – Аналитическая витрина

На основе рассчитанных коллизий сформируем аналитическую витрину, пригодную для использования в BI-системе.

Витрина будет содержать следующие агрегированные показатели:\
• количество коллизий по типам пар (A×B, A×C, B×C),\
• распределение по типам конфликтов,\
• количество критичных коллизий,\
• среднее и максимальное количество дней перекрытия,\
• распределение по статусам,\
• распределение по приоритетам.

In [ ]:
# 1. Агрегированные показатели
aggregated_metrics = {
      'Collision_Counts:': conflicts_df['pair'].value_counts().to_frame().rename(columns={'pair': 'count'}),
      'Conflict_Types (Spatial/Temporal)': conflicts_df.groupby(['spatial_intersection', 'temporal_intersection']).size().to_frame().rename(columns={0:'count'}),
      'Critical_Collisions': pd.DataFrame({'count': [conflicts_df.query("level == 'high'").shape[0]]}),    # Преобразуем в DataFrame
      'Avg_Overlap_Days': pd.DataFrame({'count': [conflicts_df['overlap_days'].mean()]}).astype(float),    # Преобразуем в DataFrame
      'Max_Overlap_Days': pd.DataFrame({'count': [conflicts_df['overlap_days'].max()]}).astype(int),    # Преобразуем в DataFrame
      'Level_Distribution:': conflicts_df['level'].value_counts().to_frame().rename(columns={'level': 'count'}),
      'Status_Distribution': conflicts_df['statuses'].explode().value_counts().to_frame().rename(columns={'statuses': 'count'}),
      'Priority_Distribution': conflicts_df['priorities'].explode().value_counts().to_frame().rename(columns={'priorities': 'count'})
}

In [ ]:
# 2. Преобразуем показатели в единый DataFrame
dashboard = pd.concat([
      aggregated_metrics['Collision_Counts:'].reset_index().assign(metric=lambda x: 'Collision Counts: ' + x['pair']),    # Используем пару вместо index
      aggregated_metrics['Conflict_Types (Spatial/Temporal)'].reset_index().assign(metric=lambda x: 'Conflict_Types (Spatial/Temporal): ' + x['spatial_intersection'].astype(str) + '/' + x['temporal_intersection'].astype(str)),
      pd.DataFrame({'metric': ['Critical_Collisions'], 'count': [aggregated_metrics['Critical_Collisions']['count'][0]] }),
      pd.DataFrame({'metric': ['Avg_Overlap_Days'], 'count': [aggregated_metrics['Avg_Overlap_Days']['count'][0]]}),
      pd.DataFrame({'metric': ['Max_Overlap_Days'], 'count': [aggregated_metrics['Max_Overlap_Days']['count'][0]] }),
      aggregated_metrics['Level_Distribution:'].reset_index().assign(metric=lambda x: 'Level Distribution: ' + x['level']),
      aggregated_metrics['Status_Distribution'].reset_index().assign(metric=lambda x: 'Status Distribution: ' + x['statuses']),
      aggregated_metrics['Priority_Distribution'].reset_index().assign(metric=lambda x: 'Priority Distribution: ' + x['priorities'])
], ignore_index=True)[['metric', 'count']]    # Оставляем только нужные столбцы

In [ ]:
# Преобразуем все значения в целочисленный тип
dashboard['count'] = dashboard['count'].astype(int, errors='ignore')

In [ ]:
# 5. Итоговая витрина
print(dashboard)

                                            metric  count
0                            Collision Counts: A-B      5
1                            Collision Counts: B-C      5
2                            Collision Counts: A-C      1
3   Conflict_Types (Spatial/Temporal): False/False      2
4    Conflict_Types (Spatial/Temporal): False/True      2
5    Conflict_Types (Spatial/Temporal): True/False      4
6     Conflict_Types (Spatial/Temporal): True/True      3
7                              Critical_Collisions      0
8                                 Avg_Overlap_Days      4
9                                 Max_Overlap_Days     10
10                         Level Distribution: low     11
11   Status Distribution: in_progress, in_progress      9
12       Status Distribution: in_progress, planned      1
13       Status Distribution: planned, in_progress      1
14                 Priority Distribution: 2.0, 1.0      4
15                 Priority Distribution: 1.0, 1.0      2
16            

In [ ]:
# Экспорт витрины в CSV
dashboard.to_csv('analytical_dashboard.csv', index=False)

Краткое описание логики витрины:

1. Collision_Counts – количество коллизий по каждой паре наборов данных (A×B, A×C, B×C);
2. Conflict_Types – виды коллизий по типу пересечения (пространственное, временное или совместное: True/False, False/True, True/True, соответственно);
3. Critical_Collisions – количество критических коллизий (уровень критичности 'high');
4. Avg_Overlap_Days – cреднее количество дней перекрытия между объектами;
5. Max_Overlap_Days – максимальное количество дней перекрытия среди всех коллизий;
6. Level_Distribution – распределение коллизий по уровням критичности ('high', 'medium', 'low'). В данном случае встречается только уровень 'low';
7. Priority_Distribution – распределение приоритетов взаимодействий (частота встречаемости высоких, средних и низких приоритетов).

При проведении анализа были учтены только работы со статусом “in progress” и “planned” как те, что могут напрямую оказывать влияние на текущие процессы.

Итоговая витрина является обобщенным результатом анализа коллизий, содержащее как количественные показатели (количества коллизий, среднее и максимальное количество дней перекрытия), так и качественные показатели (распределение по уровням критичности и приоритетам).